In [0]:
%sql
CREATE TABLE IF NOT EXISTS de_learning_workspace.silver.customers
(
    customer_id INT,

    first_name STRING,
    last_name STRING,
    email STRING,
    phone STRING,

    city STRING,
    state STRING,
    country STRING,

    created_at TIMESTAMP,
    updated_at TIMESTAMP,

    _load_timestamp TIMESTAMP,
    _source_system STRING,

    _silver_processed_timestamp TIMESTAMP
)
USING DELTA;

In [0]:
# last_modified = (
#     spark.sql("""
#         SELECT MAX(updated_at) AS updated_at
#         FROM de_learning_workspace.silver.customers
#     """)
#     .first()["updated_at"]
# )

# if last_modified is None:
#     last_modified = "1900-01-01 00:00:00"

# silver_source_df = spark.sql(f"""
# SELECT *
# FROM de_learning_workspace.bronze.customers
# WHERE updated_at > TIMESTAMP('{last_modified}')
# """)

In [0]:


# silver_source_df = spark.table(
#     "de_learning_workspace.bronze.customers"
# )

# display(silver_source_df)

In [0]:
from pyspark.sql.functions import (
    col,
    lower,
    trim,
    coalesce,
    lit,
    current_timestamp
)


silver_customers_df = (
    silver_source_df

    # remove duplicate customer records
    .dropDuplicates(["customer_id"])

    # standardize email
    .withColumn(
        "email",
        lower(trim(col("email")))
    )

    # handle missing location
    .withColumn(
        "city",
        coalesce(col("city"), lit("Unknown"))
    )

    .withColumn(
        "state",
        coalesce(col("state"), lit("Unknown"))
    )

    .withColumn(
        "country",
        coalesce(col("country"), lit("Unknown"))
    )

    # silver processing metadata
    .withColumn(
        "_silver_processed_timestamp",
        current_timestamp()
    )
)

In [0]:
# silver_customers_df.createOrReplaceTempView("silver_source")

In [0]:
# %sql
# MERGE INTO de_learning_workspace.silver.customers AS t

# USING silver_source AS s

# ON t.customer_id = s.customer_id

# WHEN MATCHED
# AND s.updated_at > t.updated_at
# THEN UPDATE SET *

# WHEN NOT MATCHED
# THEN INSERT *